**Analysis of Alveolar Type II Cell Transcriptional States in Lethal COVID-19 Lung Biopsies**

by Ashley Ramsawhook, PhD.

This work is an independent analysis of the Alveolar Type II (AT2) cell transcriptional state in lethal COVID-19 lung biopsies taken from Melms *et al*, 2021 (https://doi.org/10.1038/s41586-021-03569-1). While this notebook reproduces and modifies the pre-processing steps of the raw data performed by the authors, the exploration, analysis, deductions and conclusion are independent and self-directed. The agentic Large Language Model (LLM), Claude Sonnet 5.0 (Anthropic) was utilised for code debugging, statistical methodology discussions and literature dataset navigation. All code and markdown were written by myself.



**The Dataset**

The Melms *et al*, 2021 dataset consisted of lung biopsies extracted from 7 healthy donors and 19 COVID patients harvested post-mortem and sequencing libraries were prepared from single nuclei suspensions. Only one biopsy was harvested per donor, making the terms “Sample” and “Donor” synonymous for this dataset. Ambient RNA and debris contamination had been excluded from raw counts by the authors using CELL BENDER prior to publication of the dataset.   

**External Resources for Annotation**

A curated list of human ribosomal genes was acquired from the Broad Institute's software repository: http://software.broadinstitute.org/gsea/msigdb/download_geneset.jsp?geneSetName=KEGG_RIBOSOME&fileType=txt

**Installing Necessary Libraries and Dependencies**

In [ ]:
%pip install scanpy

In [ ]:
%pip install scvi-tools

In [ ]:
%pip install scikit-misc

In [ ]:
! pip install leidenalg

In [ ]:
! pip install gseapy

In [ ]:
! pip install celltypist --break-system-packages

In [ ]:
! pip install pydeseq2 

**Import the Necessary Libraries**

In [1]:
import pandas as pd
import numpy as np
import os
import scipy
import glob
from pathlib import Path
import seaborn as sns
from scipy.sparse import csr_matrix, issparse
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import gseapy as gp
from scipy import stats
import celltypist
from celltypist import models
import requests
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

W0920 10:00:58.610000 24760 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
e:\New virtual environment\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\New virtual environment\.venv-1\Lib\site-packages\celltypist\classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


**Setting the File Paths for Reading-in and Saving Data** 

In [2]:
from pathlib import Path

In [3]:
#Setting the input data path which contains the raw counts files
DATA_DIR = Path("E:/New Python/scRNA-seq/Tutorial/Covid scRNA-seq lung atlas/unzipped raw counts")
files = list(DATA_DIR.glob("*.csv"))

In [4]:
#Setting the output path for saving models and processed files
OUTPUT_DIR = "E:/New Python/scRNA-seq/Tutorial/Covid scRNA-seq lung atlas/processed"

**Import the Curated Geneset for Ribosomal Gene Annotation**



In [5]:
#Import the Broad Institute's Ribosomal Geneset
RIBO_LIST = Path("E:/New Python/scRNA-seq/Tutorial/Covid scRNA-seq lung atlas/KEGG_RIBOSOME.v2026.1.Hs.txt" )
ribo_genes = pd.read_table(RIBO_LIST, skiprows=2, header= None)

**Pre-Processing: Automating the Workflow**

The following function was written to compile all filtering, doublet detection and annotation steps into one workflow to be applied to all donor samples interatively. This function converts the adata object into a sparse matrix prior to any filtering or pre-processing to prevent the massive memory consumption and compute time incurred from performing it to every sample post concatenation. Minimum cell and gene filters have been applied prior to doublet detection to ensure the VAE generative deep learning model does not train on empty oil droplets or nuclei from highly stressed or dead cells in the 10X platform.

In [68]:
def pp(csv_path):

    """This function combines all filtering , pre-processing and annotation steps into one workflow
    to return an Anndata object for each sample"""

    adata = sc.read_csv(csv_path).T
    adata.X = csr_matrix(adata.X)
    adata_raw = adata.copy()
    sc.pp.filter_genes(adata, min_cells=10)
    sc.pp.filter_cells(adata, min_genes=100)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True, flavor="seurat_v3")
    scvi.model.SCVI.setup_anndata(adata)
    vae_all = scvi.model.SCVI(adata)
    vae_all.train(accelerator="gpu", enable_checkpointing=False)
    solo_all = scvi.external.SOLO.from_scvi_model(vae_all)
    solo_all.train(accelerator="gpu", enable_checkpointing=False)
    df_all = solo_all.predict()
    df_all["prediction"] = solo_all.predict(soft=False)
    df_all["diff"] = df_all.doublet - df_all.singlet
    doublets = df_all[(df_all.prediction == "doublet") & (df_all["diff"] > 0.1)]

    adata = adata_raw.copy()  # ← use the saved raw copy instead of re-reading
    sample_name = os.path.basename(csv_path).split("_")[1]
    adata.obs["Sample"] = sample_name  # ← was adata_raw, should be adata

    adata.obs["doublet"] = adata.obs.index.isin(doublets.index)
    adata = adata[~adata.obs.doublet]

    sc.pp.filter_cells(adata, min_genes=200)
    adata.var["MT"] = adata.var_names.str.startswith("MT-")
    adata.var["ribo"] = adata.var_names.isin(ribo_genes[0].values)
    sc.pp.calculate_qc_metrics(adata, qc_vars=["MT", "ribo"], percent_top=None, log1p=False, inplace=True)
    upper_lims = np.quantile(adata.obs.n_genes_by_counts.values, .98)
    adata = adata[adata.obs.n_genes_by_counts < upper_lims]
    adata = adata[adata.obs.pct_counts_MT < 20]
    adata = adata[adata.obs.pct_counts_ribo < 5]

    return adata


**Concantenate All Samples**

Once the pre-processing function has been applied to all donor samples, the samples are concatenated into one Adata object. 

In [ ]:
#Concatenate all samples into one combinbed Adata object

adatas = []

for file in files:
    adata = pp(file)
    adatas.append(adata)
    output_path = os.path.join(OUTPUT_DIR, f"{adata.obs['Sample'].iloc[0]}.h5ad")
    adata.write_h5ad(output_path)
    print(f"Saved:{output_path}")

adata_combined=sc.concat(
adatas,
join="outer",
label="batch",
keys=[a.obs["Sample"].iloc[0]for a in adatas],
index_unique="-"
)

print(adata_combined)

**Preserve the Raw Counts**

Create a new "counts" feature as a layer in the combined Adata object

In [70]:
adata_combined.layers["counts"] = adata_combined.X.copy()

**Normalise and Log Transform**

Apply normalisation and log transformation to appropriately scale the raw counts. 

In [ ]:
#Normalise the combined AnnData object 
sc.pp.normalize_total(adata_combined, target_sum=1e4)
#Check normalisation
adata_combined.X.sum(axis=1)
#Log transform the normalised values 
sc.pp.log1p(adata_combined)
#Check normalisation
adata_combined.X.sum(axis=1)

In [72]:
#Check if sparse matric
adata_combined.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 69868187 stored elements and shape (91935, 34546)>

In [ ]:
#Check the observation layer of the combined adata object
adata_combined.obs

**Save the Combined Adata Object Post-Normalisation**

In [74]:
adata_combined.write_h5ad("adata_combined.h5ad")

In [ ]:
#View the dataset to understand the dimensions and ranges of values
adata_combined.obs.groupby("Sample").count()

**Filter the Genes by Coverage to Reduce Dataset Dimensions**

Filtering the genes at this stage of the pipeline is appropriate as the full transcriptome counts has been normalised on the combined anndata object, not a subset of genes as this would skew the dataset counts scale. Filtering the genes here prior to highly variable gene filtering only removes genes which are poorly represented by cells i.e. low coverage and hence "unrepresentative." Removing the cells / nuclei with poor coverage cleans up the feature space (genes are columns and hence features in scanpy) before integration rather than the per sample QC performed prior to doublet removal.  

In [76]:
#Filter the genes with less than 100 cell coverage
sc.pp.filter_genes(adata_combined, min_cells= 100)
adata_combined

AnnData object with n_obs × n_vars = 91935 × 19980
    obs: 'Sample', 'doublet', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_MT', 'pct_counts_MT', 'total_counts_ribo', 'pct_counts_ribo', 'batch'
    var: 'n_cells'
    uns: 'log1p'
    layers: None (.X), 'counts'

There is now approximately 20,000 genes in the filtered dataset. This is an appropriate library size for integration. 

**Interogation of Highly Variable Genes in the Dataset**

 This was performed without subsetting to prevent exclusion of the non-variable genes, rendering them inaccessible for downstream analysis. Setting the limit to 3000 is two fold: The most interesting and relevant genes are usually in this space and running the interrogation on the full transcriptome would consume too much memory. As previously noted, the batch_key for individual donor discrimination is set to "Sample" as each donor only contributed one sample, thus sample and donor are synonymous. 


In [77]:
sc.pp.highly_variable_genes(
    adata_combined, n_top_genes=3000, layer="counts",
    flavor="seurat_v3", batch_key="Sample"
)

In [78]:
#Confirm adata_combined layers are populated

adata_combined

AnnData object with n_obs × n_vars = 91935 × 19980
    obs: 'Sample', 'doublet', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_MT', 'pct_counts_MT', 'total_counts_ribo', 'pct_counts_ribo', 'batch'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'log1p', 'hvg'
    layers: None (.X), 'counts'

**Sample Integration**

Sample integration was performed using SCVI tools to remove technical artifacts and batch effects between donors or "Samples."

In [80]:
#Use SCVI to perform Intergration
scvi.model.SCVI.setup_anndata(adata_combined, layer= "counts",
                              batch_key="Sample",
                              continuous_covariate_keys=["pct_counts_MT", "total_counts", "pct_counts_ribo"])

In [81]:
#Initialise the model
model_integration = scvi.model.SCVI(adata_combined)

In [ ]:
#Train the model

model_integration.train(accelerator= "gpu", enable_checkpointing= False)

In [83]:
#Save the latent representation connectivities as "X_scVI"
adata_combined.obsm["X_scVI"] = model_integration.get_latent_representation()

The X_scVI data will be used for nuclei normalised read counts based clustering using the UMAP algorithm. 

In [84]:
#Obtain the highly variable gene subset
hvg_genes = adata_combined.var_names[adata_combined.var["highly_variable"]]
normalised_hvg = model_integration.get_normalized_expression(
    adata_combined,
    gene_list = list(hvg_genes),
    library_size= 1e4
)

In [ ]:
#View highly variable genes
normalised_hvg

In [86]:
#View the nuclei connectivities 
adata_combined.obsm["X_scVI"]

array([[ 2.65971661e-01,  1.06158316e-01, -1.86395860e+00, ...,
         2.26086617e-01, -9.68161941e-01, -4.22053242e+00],
       [-2.27377892e-01,  3.49490941e-01, -2.03641701e+00, ...,
         1.14290833e+00,  1.13776267e-01, -2.60805774e+00],
       [-8.88654590e-01,  1.97305775e+00, -3.08897519e+00, ...,
         2.24808097e+00, -2.42385340e+00,  1.46970034e+00],
       ...,
       [ 6.85437560e-01,  3.76877308e-01, -2.51152539e+00, ...,
         5.42556882e-01, -7.71499991e-01, -2.05023575e+00],
       [ 8.77688527e-01,  9.14137363e-02, -1.84908533e+00, ...,
         1.85135782e+00, -1.94263458e-03, -1.05889857e+00],
       [-1.20086098e+00, -9.41604376e-03, -8.83043170e-01, ...,
        -3.05089188e+00,  8.81717801e-01,  7.12161660e-01]],
      shape=(91935, 10), dtype=float32)